## Problem Statement

Spam detection in text can often be difficult to detect for many users. Being able to detect spam messages so that they can be filtered as junk is an effective way to mitigate potentially malicious messages.

The aim of this project is to analyse potential spam messages and filter then from 'ham', none spam messages.

## Dataset Overview

The dataset used for this model is a collection of spam and non-spam messages.
Dataset link: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset

The solution of filtering spam will implement the use of machine learning tools to decipher 'spam' from 'ham'. In future, this model could be used in a web application for practical use.

In [3]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix
import joblib


## Loading the dataset

In [13]:
# the spam.csv dataset is loaded
# latin1 is used to make sure that all none standard characters are recognised
df = pd.read_csv('/Users/jameswoodthorpe/Code/University/Year_3/6033/Data/spam.csv', encoding='latin-1')
# this file path needs adapting for where the 'spam.csv' file is saved on your machine

In [5]:
# prints the dataset to confirm that it has loaded successfully
print(df.head())

     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


#### Dataset evaluations
<ul>
    <li> 1
    <li> 2
    <li> 3
    <li>
<ul>



## Cleaning the data

In [6]:
# cleans the text. firstly it is set to lower case to it can be easily parsed,
# then any punctuation and special characters are removed
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    return text

In [7]:
# text labels are mapped to numbers for the model to recognise what is 'ham' and what is 'spam'
df['v1'] = df['v1'].map({'ham': 0, 'spam': 1})
df['v2'] = df['v2'].apply(clean_text)


In [8]:
# splits data into training and testing
# stratify is used here to ensure that equal amounts of 'ham' and 'spam' are available in the sets
x_train, x_test, y_train, y_test = train_test_split(
    df['v2'],
    df['v1'],
    test_size=0.2,
    random_state=42,
    stratify=df['v1']
)

## Vectorization

In [9]:
vectorizer = TfidfVectorizer(stop_words='english')

# the model converts the text data (vocabulary) to training data
x_train_tfidf = vectorizer.fit_transform(x_train)
x_test_tfidf = vectorizer.transform(x_test)

# total number of text data
print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())}")

Vocabulary size: 6575


## Initializing and training

In [10]:
# initialize and train the data using MultinomialNB,
# this process classifies the data and calculates the probability of words appearing as 'ham' or 'spam'
model = MultinomialNB()
model.fit(x_train_tfidf, y_train)

# function used for predictions, using data the model has not seen to analyse performance
y_pred = model.predict(x_test_tfidf)

# prints the models performance for verification
# a confusion matrix is used for this verification
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[965   1]
 [ 36 113]]
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       966
           1       0.99      0.76      0.86       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.96      1115



In [11]:
# this function is used to test the model of unseen data
def predict_spam(new_message):
    # cleans the input
    cleaned_msg = clean_text(new_message)

    # vectorize the input
    vectorized_msg = vectorizer.transform([cleaned_msg])

    prediction = model.predict(vectorized_msg)

    return "SPAM" if prediction[0] == 1 else "HAM"

# unseen test inputs to verify the model succeeds
print(predict_spam("Congratulations! You've won a $500 Amazon gift card. Click here to claim."))
print(predict_spam("Hey, are we still meeting for coffee at 3pm?"))
# this input should be quite difficult for the model as it contains sentiment that might be process as spam eg. '10 minutes', '!'
print(predict_spam("Hey, I'm running about 10 minutes late for our meeting, see you soon!"))


SPAM
HAM
HAM


In [12]:
# saves the model and the vectorizer to your project folder for later use
joblib.dump(model, 'spam_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')

['vectorizer.pkl']